# Lab 2 — Data Mining

**Course:** MSc in Big Data Analytics — Data Mining  
**Institution:** Adventist University of Central Africa  
**Dataset:** UCI Credit Approval (`crx.data`)

---

## Task Objectives
1. Perform data preprocessing
2. Perform feature engineering
3. Perform hyperparameter tuning
4. Build and evaluate machine learning models

## Toolkit
`numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`

## Setup & Imports

Import required libraries and define project paths and constants.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
)
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Paths (notebook should be run from the DataMining folder)
BASE_DIR = Path.cwd()
DATA_FILE = BASE_DIR / "crx.data"
OUTPUT_CSV = BASE_DIR / "credit_approval_prepared.csv"
PLOTS_DIR = BASE_DIR / "lab2_plots"
PLOTS_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

# UCI Credit Approval attribute names (A1-A15 features, A16 target)
COLUMN_NAMES = [f"A{i}" for i in range(1, 17)]
CATEGORICAL_COLS = ["A1", "A4", "A5", "A6", "A7", "A9", "A10", "A12", "A13"]
CONTINUOUS_COLS = ["A2", "A3", "A8", "A11", "A14", "A15"]
TARGET_COL = "A16"

print(f"Data file: {DATA_FILE}")
print(f"File exists: {DATA_FILE.exists()}")

---
## 1) Data Preparation

Load the credit approval dataset into a pandas DataFrame and prepare it for analysis.

**Steps:**
- Read `crx.data` (missing values encoded as `?`)
- Clean categorical and continuous columns
- Encode the target variable (`+` = approved, `-` = rejected)
- Save the prepared DataFrame to CSV

In [ ]:
# Load comma-separated data without a header row
df = pd.read_csv(
    DATA_FILE,
    header=None,
    names=COLUMN_NAMES,
    na_values="?",
    skipinitialspace=True,
)

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# Strip whitespace from categorical columns
for col in CATEGORICAL_COLS + [TARGET_COL]:
    df[col] = df[col].astype(str).str.strip()

# Convert continuous columns to numeric
for col in CONTINUOUS_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Encode target: '+' approved (1), '-' rejected (0)
df["target"] = df[TARGET_COL].map({"+": 1, "-": 0})

print("Column data types after preparation:")
print(df.dtypes)

# Write prepared data to CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nPrepared DataFrame written to: {OUTPUT_CSV}")

---
## 2) Exploratory Data Analysis (EDA)

Use `numpy`, `pandas`, and `matplotlib` to explore the data:
- Identify missing values
- Visualize distributions and correlations
- Detect outliers using the IQR method

In [ ]:
# Missing values per attribute
missing = df[COLUMN_NAMES].isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"count": missing, "percent": missing_pct})
missing_report = missing_report[missing_report["count"] > 0].sort_values(
    "count", ascending=False
)

print("Missing values per attribute:")
display(missing_report)

total_missing_rows = df[COLUMN_NAMES].isna().any(axis=1).sum()
print(
    f"Rows with at least one missing value: {total_missing_rows} "
    f"({total_missing_rows / len(df) * 100:.1f}%)"
)

In [ ]:
# Target class distribution
class_counts = df["target"].value_counts()
print("Target class distribution:")
print(class_counts)
print((class_counts / len(df) * 100).round(2))

fig, ax = plt.subplots(figsize=(6, 4))
class_counts.plot(kind="bar", color=["#e74c3c", "#2ecc71"], ax=ax)
ax.set_title("Credit Approval Class Distribution")
ax.set_xlabel("Class (0=Rejected, 1=Approved)")
ax.set_ylabel("Count")
ax.set_xticklabels(["Rejected (-)", "Approved (+)"], rotation=0)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "01_class_distribution.png", dpi=120)
plt.show()

In [ ]:
# Histograms for continuous features
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for idx, col in enumerate(CONTINUOUS_COLS):
    df[col].dropna().hist(bins=30, ax=axes[idx], color="steelblue", edgecolor="white")
    axes[idx].set_title(f"Distribution of {col}")
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("Frequency")
plt.suptitle("Continuous Feature Distributions", y=1.02)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "02_continuous_distributions.png", dpi=120)
plt.show()

In [ ]:
# Correlation heatmap for continuous features and target
corr_cols = CONTINUOUS_COLS + ["target"]
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix (Continuous Features)")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "03_correlation_heatmap.png", dpi=120)
plt.show()

print("Correlation with target (continuous features):")
print(corr_matrix["target"].drop("target").sort_values(key=abs, ascending=False))

In [ ]:
# Outlier detection using IQR method
outlier_summary = []
for col in CONTINUOUS_COLS:
    series = df[col].dropna()
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((series < lower) | (series > upper)).sum()
    outlier_summary.append(
        {"feature": col, "outliers": n_outliers, "lower": lower, "upper": upper}
    )

outlier_df = pd.DataFrame(outlier_summary)
print("Outlier detection (IQR method):")
display(outlier_df)

# Boxplots to visualize outliers
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for idx, col in enumerate(CONTINUOUS_COLS):
    df.boxplot(column=col, ax=axes[idx])
    axes[idx].set_title(f"Boxplot of {col}")
plt.suptitle("Outlier Visualization (Boxplots)", y=1.02)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "04_outlier_boxplots.png", dpi=120)
plt.show()

---
## 3) Preprocessing, Feature Selection & Feature Engineering

Prepare the data for machine learning:
- **Outliers:** cap continuous values at IQR fences
- **Feature engineering:** create ratio, sum, and flag features
- **Missing values:** impute categoricals with mode, numerics with median
- **Label encoding:** encode categorical attributes
- **Feature selection:** SelectKBest (ANOVA F-score)
- **Scaling:** StandardScaler for model input

In [ ]:
work = df.copy()
feature_cols = CATEGORICAL_COLS + CONTINUOUS_COLS

# Cap outliers at IQR fences for continuous attributes
for col in CONTINUOUS_COLS:
    q1, q3 = work[col].quantile(0.25), work[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    work[col] = work[col].clip(lower=lower, upper=upper)

# Feature engineering
work["A2_A3_ratio"] = work["A2"] / (work["A3"] + 1)          # debt burden proxy
work["A14_A15_sum"] = work["A14"] + work["A15"]              # combined indicator
work["any_delinquency_flag"] = (
    (work["A9"].astype(str).str.lower() == "t")
    | (work["A10"].astype(str).str.lower() == "t")
).astype(int)

engineered_cols = ["A2_A3_ratio", "A14_A15_sum", "any_delinquency_flag"]
all_features = feature_cols + engineered_cols
continuous_for_model = CONTINUOUS_COLS + engineered_cols

X = work[all_features]
y = work["target"]

print(f"Feature matrix before preprocessing: {X.shape}")

In [ ]:
# Impute missing values
cat_imputer = SimpleImputer(strategy="most_frequent")
num_imputer = SimpleImputer(strategy="median")

X_imputed = X.copy()
X_imputed[CATEGORICAL_COLS] = cat_imputer.fit_transform(X[CATEGORICAL_COLS])
X_imputed[continuous_for_model] = num_imputer.fit_transform(X[continuous_for_model])

print("Missing values after imputation:", X_imputed.isna().sum().sum())

# Label encoding for categorical attributes
X_encoded = X_imputed.copy()
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_imputed[col].astype(str))

print("Label encoding applied to:", CATEGORICAL_COLS)

In [ ]:
# Feature selection: top K features by ANOVA F-score
k_best = min(12, X_encoded.shape[1])
selector = SelectKBest(score_func=f_classif, k=k_best)
X_selected = selector.fit_transform(X_encoded, y)
selected_mask = selector.get_support()
selected_features = [col for col, keep in zip(X_encoded.columns, selected_mask) if keep]

scores = pd.Series(selector.scores_, index=X_encoded.columns).sort_values(ascending=False)
print(f"Top {k_best} features selected by SelectKBest:")
display(scores.head(k_best).to_frame("f_score"))

X_final = pd.DataFrame(X_selected, columns=selected_features)

# Standard scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final)

print(f"\nFinal feature matrix shape: {X_scaled.shape}")
print("Selected features:", selected_features)

---
## 4) Model Creation and Evaluation

### 4a) Classification Model — Random Forest

Build a classification-based model using scikit-learn and evaluate with two metrics:

| Metric | Justification |
|--------|---------------|
| **Accuracy** | Measures overall correctness; easy to interpret for near-balanced classes. |
| **F1-Score** | Harmonic mean of precision and recall; important when misclassifying credit approvals/rejections has business cost. |

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=RANDOM_STATE,
    class_weight="balanced",
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print(f"Random Forest - Test Accuracy: {rf_accuracy:.4f}")
print(f"Random Forest - Test F1-Score:  {rf_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=["Rejected", "Approved"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

### 4b) Neural Network Model — 10-Fold Cross-Validation

Build a **multi-layer** neural network (`MLPClassifier` with hidden layers 64 → 32) and evaluate using **10-fold cross-validation**.

**Metrics:** Accuracy and F1-Score (same justification as above).

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=500,
    batch_size=32,
    random_state=RANDOM_STATE,
    early_stopping=True,
)

scoring = {"accuracy": "accuracy", "f1": make_scorer(f1_score)}
cv_results = cross_validate(
    mlp, X_scaled, y, cv=10, scoring=scoring, return_train_score=False
)

nn_accuracy = cv_results["test_accuracy"].mean()
nn_accuracy_std = cv_results["test_accuracy"].std()
nn_f1 = cv_results["test_f1"].mean()
nn_f1_std = cv_results["test_f1"].std()

print("Multi-layer MLP (hidden layers: 64, 32)")
print(f"10-Fold CV Accuracy: {nn_accuracy:.4f} (+/- {nn_accuracy_std:.4f})")
print(f"10-Fold CV F1-Score: {nn_f1:.4f} (+/- {nn_f1_std:.4f})")

### 4c) Hyperparameter Tuning — Grid Search + 10-Fold CV

Use `GridSearchCV` to find the best hyperparameter combination. The search space includes **at least three hyperparameters**:

- `learning_rate_init` (learning rate)
- `max_iter` (epochs)
- `activation` (activation function)
- `batch_size`
- `solver` (optimizer)

> **Note:** Grid search may take several minutes to complete.

In [ ]:
base_mlp = MLPClassifier(random_state=RANDOM_STATE, early_stopping=True)

param_grid = {
    "hidden_layer_sizes": [(64, 32), (128, 64)],
    "activation": ["relu", "tanh"],
    "learning_rate_init": [0.001, 0.01, 0.05],
    "batch_size": [16, 32, 64],
    "max_iter": [300, 500],
    "solver": ["adam", "sgd"],
}

grid_search = GridSearchCV(
    estimator=base_mlp,
    param_grid=param_grid,
    cv=10,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
)

print("Running Grid Search (this may take a few minutes)...")
grid_search.fit(X_scaled, y)

print("\nBest hyperparameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest 10-Fold CV F1-Score: {grid_search.best_score_:.4f}")

In [ ]:
# Evaluate tuned model with both metrics via 10-fold CV
best_model = grid_search.best_estimator_
tuned_cv = cross_validate(
    best_model,
    X_scaled,
    y,
    cv=10,
    scoring={"accuracy": "accuracy", "f1": make_scorer(f1_score)},
)

tuned_accuracy = tuned_cv["test_accuracy"].mean()
tuned_f1 = tuned_cv["test_f1"].mean()

print(f"Tuned MLP - 10-Fold CV Accuracy: {tuned_accuracy:.4f}")
print(f"Tuned MLP - 10-Fold CV F1-Score:  {tuned_f1:.4f}")

### 4d) Model Comparison

The assignment asks to compare **the model in 4a** against **the model in 4b**, using visualizations.

Because section **4c** produces the **tuned** neural network, we provide **two comparisons**:

| Comparison | Models | Purpose |
|------------|--------|---------|
| **Primary (per 4d wording)** | Random Forest **(4a)** vs Baseline MLP **(4b)** | Direct comparison requested between the classification model and the neural network from 4b |
| **Supplementary** | Random Forest **(4a)** vs Tuned MLP **(4c)** | Shows whether hyperparameter tuning (4c) improves the neural network relative to the classification model |

Both comparisons use **Accuracy** and **F1-Score**, with bar charts for visual clarity.

In [ ]:
# Comparison 1 (4d primary): classification model (4a) vs baseline neural network (4b)
comparison_4a_vs_4b = pd.DataFrame(
    {
        "Model": ["Random Forest (4a)", "Baseline MLP (4b)"],
        "Accuracy": [rf_accuracy, nn_accuracy],
        "F1-Score": [rf_f1, nn_f1],
    }
)

# Comparison 2 (supplementary): classification model (4a) vs tuned neural network (4c)
comparison_4a_vs_4c = pd.DataFrame(
    {
        "Model": ["Random Forest (4a)", "Tuned MLP (4c)"],
        "Accuracy": [rf_accuracy, tuned_accuracy],
        "F1-Score": [rf_f1, tuned_f1],
    }
)

print("4d Primary comparison — 4a vs 4b:")
display(comparison_4a_vs_4b)

print("\nSupplementary comparison — 4a vs 4c (after hyperparameter tuning):")
display(comparison_4a_vs_4c)

In [ ]:
def plot_model_comparison(comparison_df, title, filename):
    """Bar charts for Accuracy and F1-Score between two models."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    x = np.arange(len(comparison_df))

    for i, metric in enumerate(["Accuracy", "F1-Score"]):
        axes[i].bar(x, comparison_df[metric], color=["#3498db", "#9b59b6"], width=0.45)
        axes[i].set_xticks(x)
        axes[i].set_xticklabels(comparison_df["Model"], rotation=15, ha="right")
        axes[i].set_ylim(0, 1)
        axes[i].set_title(f"{metric}")
        axes[i].set_ylabel(metric)
        for j, val in enumerate(comparison_df[metric]):
            axes[i].text(j, val + 0.01, f"{val:.3f}", ha="center", fontsize=10)

    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / filename, dpi=120)
    plt.show()


# Primary 4d visualization: 4a vs 4b
plot_model_comparison(
    comparison_4a_vs_4b,
    title="4d Comparison: Random Forest (4a) vs Baseline MLP (4b)",
    filename="05_comparison_4a_vs_4b.png",
)

In [ ]:
# Supplementary visualization: 4a vs tuned MLP from 4c
plot_model_comparison(
    comparison_4a_vs_4c,
    title="Supplementary: Random Forest (4a) vs Tuned MLP (4c)",
    filename="06_comparison_4a_vs_4c.png",
)

---
## Summary

| Step | Output |
|------|--------|
| Data preparation | `credit_approval_prepared.csv` |
| EDA | Missing value report, distributions, outlier analysis |
| Preprocessing | Imputation, label encoding, feature engineering, scaling |
| 4a — Random Forest | Accuracy & F1 on held-out test set |
| 4b — Baseline MLP | 10-fold CV Accuracy & F1 |
| 4c — Tuned MLP | Grid Search best params + 10-fold CV metrics |
| 4d — Comparison | `05_comparison_4a_vs_4b.png` (primary) and `06_comparison_4a_vs_4c.png` (supplementary) |

In [ ]:
print("Pipeline complete.")
print(f"  Prepared CSV : {OUTPUT_CSV}")
print(f"  Plots folder : {PLOTS_DIR}")
print()
print(f"  Random Forest      -> Accuracy: {rf_accuracy:.4f}, F1: {rf_f1:.4f}")
print(f"  Baseline MLP (CV)  -> Accuracy: {nn_accuracy:.4f}, F1: {nn_f1:.4f}")
print(f"  Tuned MLP (CV)     -> Accuracy: {tuned_accuracy:.4f}, F1: {tuned_f1:.4f}")